# Samudra Prahari — Colab training template (T0.3)
Standard flow: mount Drive → clone repo → install → pull manifests → train → export ONNX → metrics.json → register.
Copy this per model (autoencoder, YOLO11n, UNet). Replace the TRAIN cell with the real recipe.

In [ ]:
# 1. Mount Drive (weights + data persist here)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Clone repo + install editable
%cd /content
!git clone https://github.com/ASugandhan/SamudraPrahari.git || (cd SamudraPrahari && git pull)
%cd SamudraPrahari
!pip install -q -e '.[models]'

In [ ]:
# 3. Pull manifests / data (from Drive or fetch.py)
!ls data/manifests || python scripts/data/fetch.py all
MODEL_NAME = 'dummy'   # <- set: stage_a_ae | yolo11n | unet
OUT = f'models/{MODEL_NAME}.onnx'

In [ ]:
# 4. TRAIN (dummy stand-in — replace with the real recipe per model)
import torch, torch.nn as nn
model = nn.Sequential(nn.Conv2d(1, 4, 3, padding=1), nn.ReLU(), nn.Conv2d(4, 1, 3, padding=1))
model.eval()
# ... real training loop writes val metrics into `metrics` ...
metrics = {'model': MODEL_NAME, 'note': 'dummy run', 'val_metric': None}

In [ ]:
# 5. Export ONNX (opset 13, dynamic batch) + metrics.json
import json
dummy = torch.randn(1, 1, 64, 64)
torch.onnx.export(model, dummy, OUT, opset_version=13,
                  input_names=['x'], output_names=['y'],
                  dynamic_axes={'x': {0: 'b'}, 'y': {0: 'b'}})
json.dump(metrics, open(f'models/{MODEL_NAME}_metrics.json', 'w'), indent=2)
print('exported', OUT)

In [ ]:
# 6. Validate on CPU + register (updates models/registry.json)
!python scripts/register_model.py --onnx {OUT} --name {MODEL_NAME} \
    --metrics models/{MODEL_NAME}_metrics.json
# 7. Copy weights back to Drive
!cp models/{MODEL_NAME}.onnx /content/drive/MyDrive/